### Create the png and the different fits of the images being too zoomed in from the size estimate made on the 6" resolution

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

import os

from astropy import units as u
from astropy.io import fits
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy.table import Table, vstack
from astropy.wcs import WCS
from astropy.nddata import Cutout2D

In [2]:
def extract(FITS):
    '''Extracts the information needed from a FITS format catalogue:
        - Coordinates
        - Facets
        - ID
        - Coordinates of the optical host'''

    sources_ID = ['ILTJ161851.16+540420.78', 'ILTJ161757.09+545112.84', 'ILTJ160757.86+541634.95', 'ILTJ161212.30+552303.74', 'ILTJ160736.53+535731.25', 'ILTJ160328.90+540544.61', 'ILTJ161537.88+534646.43', 'ILTJ161027.78+541242.46', 'ILTJ161040.42+540630.05', 'ILTJ161756.90+545113.97', 'ILTJ160604.57+561046.10', 'ILTJ160552.90+561045.24', 'ILTJ161704.37+545023.63', 'ILTJ160757.43+551015.48', 'ILTJ161743.53+553526.75', 'ILTJ160252.85+545955.11', 'ILTJ160714.04+544903.18', 'ILTJ161000.69+535243.46', 'ILTJ160603.03+534812.71', 'ILTJ161411.89+554133.79', 'ILTJ160847.89+540519.88']
    coords = []
    facets = []
    opt_RADEC = []
    
    with fits.open(FITS) as hdul:
        cat = Table(hdul[1].data)
    
        for i in sources_ID:
            for j in range(len(cat['Source_ID'])):
                if i == cat['Source_ID'][j]:
                    coords.append(SkyCoord(ra=cat['RA'][j], dec=cat['DEC'][j], unit=(u.deg, u.deg)))
                    facets.append(cat['facet'][j])
                    opt_RADEC.append((cat['opt_RA'][j], cat['opt_DEC'][j]))
                
    return coords, facets, sources_ID, opt_RADEC

In [3]:
def get_rms(image):
    '''Return the noise mesure of an image'''
    
    maskSup = 1e-6
    m = image[np.abs(image) > maskSup]
    rmsold = np.std(m)
    diff = 0.1
    cut = 3.0
    med = np.median(m)
    
    for _ in range(10):
        m_filtered = m[np.abs(m - med) < rmsold * cut]
        rms = np.std(m_filtered)
        if np.abs((rms - rmsold) / rmsold) < diff:
            break
        rmsold = rms
        
    return rms

In [4]:
def make_Mingo_cutout(FITS, pos):
    '''Cuts an image from a FITS around a source with an estimated size'''
    
    with fits.open(FITS) as image_hdul:
        image = image_hdul[0].data
        image = np.squeeze(image)
        
        init_size = 50 # First size to reduce the image to a small square
        threshold = 35 # Coefficient to be applied to the noise measurement to define the minimum value of the source pixels
        margin = 10 # Pixel margin around the source
        
        image_header = image_hdul[0].header
        wcs = WCS(image_header, naxis=2)
        cutout = Cutout2D(image, pos, size=init_size, wcs=wcs) # First cutout
        
    rms = get_rms(cutout.data) # Noise measurement
    threshold = threshold * rms
    source = np.argwhere(cutout.data > threshold) 

    size = init_size
    if source.size > 0:
        y, x = source[:, 0], source[:, 1]
        center = cutout.data.shape[0]//2
        
        size = 2*max(center - np.min(x) + margin, np.max(x) + margin - center,
                   center - np.min(y) + margin, np.max(y) + margin - center
                  ) # Size estimation of the source in pixel
        
        cutout = Cutout2D(cutout.data, pos, size=size, wcs=cutout.wcs) # Second cutout

    pixel_scale = np.abs(cutout.wcs.wcs.cdelt[0])
    degree_size = size*pixel_scale # Size estimation of the source in degree

    return cutout.data, image_header, cutout.wcs, degree_size

In [5]:
def make_sizebased_cutout(FITS, pos, size, label): 
    '''Cuts an image from a FITS around a source with a known size'''
    
    with fits.open(FITS) as image_hdul:
        image = image_hdul[0].data

        if label == 'Mingo':
            image = np.squeeze(image)
        
        image_header = image_hdul[0].header
        wcs = WCS(image_header, naxis=2)
        print
        cutout = Cutout2D(image, pos, size=size, wcs=wcs) # Cutout
        
    return cutout.data, image_header, cutout.wcs

In [6]:
def plot_cutouts(cutouts, output_filename):
    '''Create a png with 4 images next to each other'''
    
    fig = plt.figure(figsize=(18, 6))
    gs = fig.add_gridspec(1, len(cutouts), wspace=0.3)

    for i, (cutout, wcs, label, opt_RADEC) in enumerate(cutouts):
        ax = fig.add_subplot(gs[0, i], projection=wcs)
        cutout = np.nan_to_num(cutout, nan=0.0)
        
        if label == 'Mingo':
            vmax = cutout.max()
            vmin = 0
        elif label == '12"':
            vmin = get_rms(cutout)
            vmax = cutout.max()
        elif label == '06"':
            vmin = get_rms(cutout)
            vmax = cutout.max()
        else:
            vmin = get_rms(cutout)
            vmax = cutout.max()
            
        im = ax.imshow(cutout, norm=PowerNorm(vmin=vmin, gamma=0.6), cmap='turbo', origin='lower')
        cbar = plt.colorbar(im, ax=ax, orientation='horizontal', shrink=0.7, label='JY/BEAM')
        cbar.set_ticks([vmin, (vmin+vmax)/2, vmax])
        ax.set_xlabel('RA')
        ax.set_ylabel('DEC')
        ax.set_title(f'{label}')

        ra, dec = opt_RADEC
        sky_coord = SkyCoord(ra=ra, dec=dec, unit=(u.deg, u.deg), frame='icrs')
        x_pixel, y_pixel = wcs.world_to_pixel(sky_coord)
        ax.plot(x_pixel, y_pixel, '+', color='white', markersize=15, mew=2) # Creates a white cross on images representing optical host

    fig.subplots_adjust(left=0.06, right=0.99, top=0.85, bottom=0.1)

    plt.savefig(output_filename, format='png')
    plt.close(fig)

In [7]:
def create_fits_image_cutout(output_folder, catalogue):
    '''Create a png with 4 images (one image per resolution) next to each other and the fits corresponding to each image.'''
    
    os.makedirs(output_folder, exist_ok=True)
    sources_coords, sources_facets, sources_ID, opt_RADEC = extract(catalogue)
  
    for i, pos in enumerate(sources_coords):# View all sources
        output_filename = os.path.join(output_folder, f'{sources_ID[i]}')
        file_paths = [
            ('Mingo', f'image_full_ampphase_di_m.NS_shift.int.facetRestored.blanked.scaled-crop.fits'),
            ('12', f'./_12_Facets/facet_{sources_facets[i]}.fits'),
            ('06', f'./_06_Facets/facet_{sources_facets[i]}.fits'),
            ('03', f'./_03_Facets/facet_{sources_facets[i]}.fits')
        ]

        cutouts = []
        for label, file_path in file_paths: # Create the cutouts for each resolutions
            if label == 'Mingo':
                cutout, header, wcs, degree_size = make_Mingo_cutout(file_path, pos)

            else:
                cutout, header, wcs = make_sizebased_cutout(file_path, pos, degree_size*u.deg, label)

            '''Create FITS files'''
            header.update(wcs.to_header())
            header['NAXIS1']=cutout.shape[1]
            header['NAXIS2']=cutout.shape[0]
            hdu = fits.PrimaryHDU(cutout, header=header)
            hdul = fits.HDUList([hdu])
            hdul.writeto(f'{output_filename}_{label}.fits', overwrite=True)
            
            cutouts.append((cutout, wcs, label, opt_RADEC[i]))
        
        plot_cutouts(cutouts, f'{output_filename}.png') # Create the pngs 

In [8]:
create_fits_image_cutout('Final_facet_subimages', 'cat_classified.fits')